# Exercise 10: Webscraping with beautifulsoup4

> Note: to import the python package `bs4` (beautifulsoup4) in the cell below, bs4 must be installed on your computer. We learned how to do this in Exercise09. See the [exercise09_conda](../../unit09/exercise09/exercise09_conda.md) for detailed instructions!

In [ ]:
# import packages we need
import requests
import bs4 # this is beautifulsoup4
import random
random.seed(42) # change to your personalized seed if you want

## Links to Michael from Michael's links

Find out **how many of the websites linked on Michael's homepage ([http://michael.szell.net](http://michael.szell.net)) link back to his homepage**? For example, the very first link on his website is [https://en.itu.dk](https://en.itu.dk) - now you need to check whether the website [https://en.itu.dk](https://en.itu.dk) contains the link [http://michael.szell.net](http://michael.szell.net); if yes, that increases the count of linking-back websites by 1.

For this task, you need to
1. get the content of http://michael.szell.net using `requests`
2. search the content for all links to external websites using `beautifulsoup4`
3. for each of the links from step 2, 
    * get the content with `requests`
    * search the content for all links with `beautifulsoup`
    * check whether any of the links contains `michael.szell.net`

> Note: When creating your list of links on Michael's website (step 2), remove the links indicated below from the list to avoid connection time out errors.

Extra challenge: "What does this have to do with Google?" >> Read up here: [PageRank algorithm](https://en.wikipedia.org/wiki/PageRank)

In [ ]:
# from the link list in step 2, remove all links containig these strings:
links_to_remove = [
    "www.datainterfaces.org",
    "lab.moovel.com",
    "senseable.mit.edu",
    "www.complex-systems.com",
    "ceu.edu",
    "www.complex-systems.meduniwien.ac.at"
]

In [ ]:
# get the data with the "requests" module
response = requests.get("http://michael.szell.net")
# the html code is in the attribute .content
my_text = response.content
# make your "soup" (use bs4 to read the html code)
soup_michael = bs4.BeautifulSoup(my_text)
# now we have the html code in the "soup" variable,
# this "soup" variable can be easily searched with bs4 functions.
print(soup_michael)

In [ ]:
# find all the html objects that contain links on Michael's website
all_links = [l for l in soup_michael.find_all("a")]
all_links

In [ ]:
# extract from all the links only the hyperlinks with .get("href")
all_hyperlinks = [l.get("href") for l in all_links if l.get("href")]

In [ ]:
# keep only the exteral links to websites - the ones that start with "http", and don't contain michael.szell
all_external_links = [link for link in all_hyperlinks if (not "michael.szell" in link) and (link[0:4]=="http")]
# all_external_links

In [ ]:
# remove the indicatd links (to avoid connection timeout)
for link in links_to_remove:
    all_external_links = [l for l in all_external_links if link not in l]
print(len(all_external_links))

**Now we need to webscrape each of the websites in `all_external_links`; find the links on each of those websites; and find out whether any of them contain the string `michael.szell.net`**

In [ ]:
# function that, given a website, scrapes it for its hyperlinks;
# and returns a list of only those hyperlinks that contain a specified string.
# we will call this function on each of the links on Michael's website.
def find_link_on_page(my_page, my_string):
    '''
    takes a website (my_page; url) and a string (my_string) as input;
    webscrapes my_page and checks whether any of its links contain my_string;
    returns the list of links on my_page that contain my_string 
    '''
    # get the contents of my_page and make a "soup"
    my_page_response = requests.get(my_page)
    my_page_text = my_page_response.text
    soup=bs4.BeautifulSoup(my_page_text)

    # search the contents of the page for links
    # store all links on the page in the variable all_links
    all_links = [l for l in soup.find_all("a")]

    # extract only the hyperlinks ("href" in html code)
    links_href = [l.get("href") for l in all_links if l.get("href")]

    # make a list of only those hyperlinks that contain my_string
    links_found = [l for l in links_href if my_string in l]

    return links_found

In [ ]:
# initiate a list
websites_linking_to_michael = []

# loop through all the external links
for l in all_external_links:
    # at each step, if the website of the link contains "michael.szell.net" in its links
    links_found = find_link_on_page(l, "michael.szell.net")
    # if so (if not an empty list is returned),
    # add the current link to our list:
    if links_found:
        websites_linking_to_michael.append(l)
        # and print it out
        print(l)

In [ ]:
# How many links link back to michael? # Count only the unique ones:
set(websites_linking_to_michael)
# 6! (or less, if you don't count pages/subpages as separate websites)